In [ ]:
#RSA#
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr, pearsonr

from typing import Literal
import numpy as np

class RepresentationalSimilarityAnalysis:
    """
    Representational Similarity Analysis (RSA).

    Given two representation matrices X and Y with the same number of conditions
    (rows), RSA:

    1. Computes a Representational Dissimilarity Matrix (RDM) for each:
       RDM_X[i, j] = dissimilarity(x_i, x_j)
       RDM_Y[i, j] = dissimilarity(y_i, y_j)

    2. Flattens the upper triangles of both RDMs and computes a correlation
       between them (Pearson or Spearman).
    """

    def __init__(
        self,
        dissimilarity: Literal["correlation", "euclidean", "cosine"] = "correlation",
        similarity_metric: Literal["pearson", "spearman"] = "spearman",
    ):
        self.dissimilarity = dissimilarity
        self.similarity_metric = similarity_metric

    def __call__(self, X: np.ndarray, Y: np.ndarray) -> float:
        """
        Compute RSA similarity between X and Y.

        Parameters
        ----------
        X, Y : np.ndarray
            Arrays of shape (n_conditions, ...) that may need to be flattened
            along feature dimensions.

        Returns
        -------
        rsa_similarity : float
            Correlation between the vectorized upper triangles of the two RDMs.
        """
        return self.forward(X, Y)

    def forward(self, X: np.ndarray, Y: np.ndarray) -> float:
        # Ensure both matrices have the same number of stimuli/conditions
        if X.shape[0] != Y.shape[0]:
            raise ValueError(
                f"X and Y must have the same number of conditions (rows). "
                f"Got X: {X.shape[0]}, Y: {Y.shape[0]}"
            )
            
        # 1. Compute RDMs for both spaces
        rdm_x = self.compute_rdm(X)
        rdm_y = self.compute_rdm(Y)
        
        # 2. Compare the two RDMs
        return self.compare_rdms(rdm_x, rdm_y)

        
    def compute_rdm(self, X: np.ndarray) -> np.ndarray:
        """
        Compute the Representational Dissimilarity Matrix (RDM)
        for a given representation matrix X.

        Parameters
        ----------
        X : np.ndarray
            Array of shape (n_conditions, n_features).

        Returns
        -------
        rdm : np.ndarray
            Array of shape (n_conditions, n_conditions) with pairwise dissimilarities.
        """
        # Flatten any spatial/temporal dimensions, keeping n_conditions as the first axis
        X_flat = X.reshape(X.shape[0], -1)
        
        # pdist computes the pairwise distances; squareform converts it to a symmetric matrix
        condensed_distance = pdist(X_flat, metric=self.dissimilarity)
        rdm = squareform(condensed_distance)
        
        return rdm
    def compare_rdms(self, rdm1: np.ndarray, rdm2: np.ndarray) -> float:
        """
        Compare two RDMs by correlating their upper triangles.
        """
        # Extract indices for the upper triangle, excluding the diagonal (k=1)
        row_idx, col_idx = np.triu_indices(rdm1.shape[0], k=1)
        
        # Vectorize the upper triangles
        vec1 = rdm1[row_idx, col_idx]
        vec2 = rdm2[row_idx, col_idx]
        
        # Compute the specified correlation metric
        if self.similarity_metric == "spearman":
            correlation, _ = spearmanr(vec1, vec2)
        elif self.similarity_metric == "pearson":
            correlation, _ = pearsonr(vec1, vec2)
        else:
            raise ValueError(f"Unsupported similarity metric: {self.similarity_metric}")
            
        return float(correlation)

In [1]:
#CKA#
import numpy as np

class CenteredKernelAlignment:
    """
    Unbiased linear CKA only.

    Parameters
    ----------
    eps : float
        Small constant for numerical stability.
    dtype : np.dtype
        Data type used for computations.
    """

    def __init__(
        self,
        eps: float = 1e-8,
        dtype: np.dtype = np.float64,
    ):
        self.eps = eps
        self.dtype = dtype  

    def __call__(self, X: np.ndarray, Y: np.ndarray) -> float:
        return self.forward(X, Y)

    def forward(self, X: np.ndarray, Y: np.ndarray) -> float:
        X = np.asarray(X).astype(self.dtype)
        Y = np.asarray(Y).astype(self.dtype)

        if X.shape[0] != Y.shape[0]:
            raise ValueError(
                f"Batch sizes must match along axis 0: {X.shape[0]} vs {Y.shape[0]}"
            )

        # Flatten to (n_samples, n_features)
        X = X.reshape(X.shape[0], -1)
        Y = Y.reshape(Y.shape[0], -1)

        return self._unbiased_linear_cka(X, Y)

    def _unbiased_linear_hsic(self, X: np.ndarray, Y: np.ndarray) -> float:
        """
        Unbiased HSIC estimator for the linear kernel.

        X : [n, d_x]
        Y : [n, d_y]
        """
        n = X.shape[0]
        if n <= 3:
            raise ValueError("Unbiased HSIC requires at least 4 samples (n > 3).")

        # Form linear Gram matrices
        K = X @ X.T
        L = Y @ Y.T

        # Set diagonals to zero to remove the bias term
        np.fill_diagonal(K, 0)
        np.fill_diagonal(L, 0)

        # Calculate the three terms of the unbiased HSIC estimator
        # 1. Trace of product: tr(K L)
        term1 = np.sum(K * L) 
        
        # 2. Product of sums: (1^T K 1 * 1^T L 1) / ((n-1)*(n-2))
        term2 = (np.sum(K) * np.sum(L)) / ((n - 1) * (n - 2))
        
        # 3. Dot product of row sums: (1^T K L 1) / (n-2)
        term3 = 2 * np.dot(np.sum(K, axis=1), np.sum(L, axis=1)) / (n - 2)

        hsic = (term1 + term2 - term3) / (n * (n - 3))
        
        return hsic

    def _unbiased_linear_cka(self, X: np.ndarray, Y: np.ndarray) -> float:
        """
        Unbiased linear CKA:

            CKA_unb(X, Y) =
                HSIC_unb(X, Y) / sqrt(HSIC_unb(X, X) * HSIC_unb(Y, Y))
        """
        hsic_xy = self._unbiased_linear_hsic(X, Y)
        hsic_xx = self._unbiased_linear_hsic(X, X)
        hsic_yy = self._unbiased_linear_hsic(Y, Y)

        # Bound the denominator components at 0 to avoid NaNs from floating point
        # inaccuracies in edge cases where HSIC(X,X) might be infinitesimally negative.
        denom = np.sqrt(max(hsic_xx * hsic_yy, 0.0)) + self.eps
        
        return hsic_xy / denom

In [ ]:
### WORKS::::: EEG, each time point, CUDA, N_TRAIN = 7500, PER CH

# ============================================================
# SECTION 2.4 — Predictive Alignment: Linear Encoding Models
#               EEG-ONLY VERSION
# ============================================================

# ─────────────────────────────────────────────────────────────
# CELL 2.4.0  ·  Layer discovery & shared loading utilities
# ─────────────────────────────────────────────────────────────
import h5py, os, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import explained_variance_score
from scipy.stats import pearsonr as scipy_pearsonr
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

DATA_DIR = "/shared/NX-414/data/"
FEAT_DIR = "/shared/NX-414/extracted_features/"

MODEL_A = "adv_resnet152_imagenet_full_ffgsm_eps-1_alpha-125-ep10_seed-0"
MODEL_B = "Qwen3-VL-2B-Instruct"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

EEG_SUB, EEG_ROI = "sub-01", "occipital_parietal"
EEG_TIME_MS = 100          # target time point in milliseconds

def list_h5_layers(path: str) -> list[str]:
    layers = []
    def _visit(name, obj):
        if isinstance(obj, h5py.Dataset) and name != "ids":
            layers.append(name)
    with h5py.File(path, "r") as f:
        f.visititems(_visit)
    return sorted(layers)

def h5_indexed_read(path: str, dataset_key: str, indices: np.ndarray) -> np.ndarray:
    sort_order    = np.argsort(indices)
    restore_order = np.argsort(sort_order)
    sorted_idx    = indices[sort_order]
    with h5py.File(path, "r") as f:
        data = f[dataset_key][sorted_idx.tolist(), :]
    return data[restore_order]

def make_index_map(feat_path: str) -> dict:
    with h5py.File(feat_path, "r") as f:
        ids = f["ids"][:]
    return {v: i for i, v in enumerate(ids)}

def get_feat_indices(feat_path: str, stimulus_ids: np.ndarray) -> np.ndarray:
    idx_map = make_index_map(feat_path)
    return np.array([idx_map[s] for s in stimulus_ids])


# ─────────────────────────────────────────────────────────────
# CELL 2.4.1  ·  Load EEG neural train/test data
# ─────────────────────────────────────────────────────────────
# CELL 2.4.1 · Load EEG neural train/test data — ALL TIMEPOINTS

eeg_ids = {}
eeg_neural, eeg_nc = {}, {}
with h5py.File(os.path.join(DATA_DIR, "things_eeg2.h5"), "r") as f:
    eeg_ids["train"] = f["train/stimulus_ids"][:]
    eeg_ids["test"]  = f["test/stimulus_ids"][:]
    raw_train = f[f"train/neural_data/{EEG_SUB}/{EEG_ROI}"][:]  # (16540, C, T)
    raw_test  = f[f"test/neural_data/{EEG_SUB}/{EEG_ROI}"][:]   # (200, C, T)
    nc_2d     = f[f"noise_ceilings/{EEG_SUB}/{EEG_ROI}"][:]     # (C, T)

EEG_FS = 100  # Hz
N_CH, N_T = raw_train.shape[1], raw_train.shape[2]

# Keep full (stimuli, C, T) — no time slicing
eeg_neural["train"] = raw_train               # (16540, C, T)
eeg_neural["test"]  = raw_test                # (200, C, T)
eeg_nc_2d           = nc_2d                   # (C, T)  — keep 2D

# Time axis in ms (for plotting later)
time_ms = np.arange(N_T) * (1000 / EEG_FS)    # 0, 10, 20, ..., 790

print(f"EEG2  — {EEG_SUB}/{EEG_ROI} | ALL timepoints | "
      f"train={eeg_neural['train'].shape}, test={eeg_neural['test'].shape}, "
      f"nc={eeg_nc_2d.shape}")


# ─────────────────────────────────────────────────────────────
# CELL 2.4.2  ·  Discover layers & build stimulus→feature maps
# ─────────────────────────────────────────────────────────────

path_a_things = os.path.join(FEAT_DIR, MODEL_A, "things_stimuli.h5")
path_b_things = os.path.join(FEAT_DIR, MODEL_B, "things_stimuli.h5")

layers_a = list_h5_layers(path_a_things)
layers_b = list_h5_layers(path_b_things)
print(f"Model A: {len(layers_a)} layers  |  Model B: {len(layers_b)} layers")

feat_idx_eeg_train = get_feat_indices(path_a_things, eeg_ids["train"])
feat_idx_eeg_test  = get_feat_indices(path_a_things, eeg_ids["test"])

print("Stimulus → feature index maps built ✓")


# ─────────────────────────────────────────────────────────────
# CELL 2.4.3  ·  Metric functions
# ─────────────────────────────────────────────────────────────

def pearson_r_per_unit(Y_true: np.ndarray, Y_pred: np.ndarray) -> np.ndarray:
    n  = Y_true.shape[0]
    yt = Y_true - Y_true.mean(0)
    yp = Y_pred - Y_pred.mean(0)
    num = (yt * yp).sum(0)
    den = np.sqrt((yt ** 2).sum(0) * (yp ** 2).sum(0)) + 1e-12
    return num / den

def explained_variance_per_unit(Y_true: np.ndarray, Y_pred: np.ndarray) -> np.ndarray:
    ss_res = ((Y_true - Y_pred) ** 2).sum(0)
    ss_tot = ((Y_true - Y_true.mean(0)) ** 2).sum(0) + 1e-12
    return 1.0 - ss_res / ss_tot

def nc_correct_r(r: np.ndarray, nc_ev: np.ndarray) -> np.ndarray:
    nc_r = np.sqrt(np.clip(nc_ev, 1e-6, None))
    return r / nc_r

def nc_correct_ev(ev: np.ndarray, nc_ev: np.ndarray) -> np.ndarray:
    return ev / np.clip(nc_ev, 1e-6, None)

def summarise_metrics(r, r_nc, ev, ev_nc, nc_flat, label="") -> dict:
    """Works on 1D arrays of length C*T (channels × timepoints flattened)."""
    mask = nc_flat > 10.0
    def _m(arr):
        if mask.sum() == 0:
            return float(np.nanmean(arr))
        return float(np.nanmean(arr[mask]))
    return {
        "pearsonr":              _m(r),
        "pearsonr_nc":           _m(r_nc),
        "explained_variance":    _m(ev),
        "explained_variance_nc": _m(ev_nc),
    }


# ─────────────────────────────────────────────────────────────
# CELL 2.4.4  ·  Ridge encoding model with validation-set alpha
# ─────────────────────────────────────────────────────────────

# Weight-decay grid (L2 regularization strength for Adam)
WD_GRID = np.logspace(-3, 3, 13)   # 0.001 … 1000

def fit_ridge_encoding(X_train, Y_train, val_frac=0.15, seed=42, alpha_grid=None, **kwargs):
    if alpha_grid is None:
        alpha_grid = np.logspace(0, 7, 15)

    rng     = np.random.default_rng(seed)
    n       = X_train.shape[0]
    n_val   = max(1, int(n * val_frac))
    val_idx = rng.choice(n, n_val, replace=False)
    tr_idx  = np.setdiff1d(np.arange(n), val_idx)

    Xtr, Ytr = X_train[tr_idx], Y_train[tr_idx]
    Xvl, Yvl = X_train[val_idx], Y_train[val_idx]

    Xtr_t = torch.from_numpy(Xtr.astype(np.float32)).to(DEVICE)
    mean  = Xtr_t.mean(0, keepdim=True)
    std   = Xtr_t.std(0, keepdim=True).clamp(min=1e-8)
    Xtr_t = (Xtr_t - mean) / std
    Xvl_t = (torch.from_numpy(Xvl.astype(np.float32)).to(DEVICE) - mean) / std

    # ─── CENTER Y ───
    Ytr_t  = torch.from_numpy(Ytr.astype(np.float32)).to(DEVICE)
    y_mean = Ytr_t.mean(0, keepdim=True)             # (1, C)
    Ytr_c  = Ytr_t - y_mean                           # centered for fitting
    # ────────────────

    N = Xtr_t.shape[0]
    K  = Xtr_t @ Xtr_t.T
    Kv = Xvl_t @ Xtr_t.T

    eigvals, U = torch.linalg.eigh(K)
    UtY = U.T @ Ytr_c                                 # use centered Y
    KvU = Kv @ U

    best_alpha, best_score = alpha_grid[0], -np.inf
    for alpha in alpha_grid:
        t0 = time.time()
        inv = 1.0 / (eigvals + float(alpha))
        Yv_hat = (KvU * inv) @ UtY + y_mean           # ADD MEAN BACK
        score = float(np.nanmean(pearson_r_per_unit(Yvl, Yv_hat.cpu().numpy())))
        marker = " ◀ best so far" if score > best_score else ""
        print(f"        α={alpha:.1e}  val_r={score:.4f}{marker}  ({time.time()-t0:.2f}s)")
        if score > best_score:
            best_score, best_alpha = score, alpha

    del Xtr_t, Xvl_t, Ytr_t, Ytr_c, K, Kv, eigvals, U, UtY, KvU
    torch.cuda.empty_cache()

    # Refit on full train
    X_full_t = torch.from_numpy(X_train.astype(np.float32)).to(DEVICE)
    mean_f   = X_full_t.mean(0, keepdim=True)
    std_f    = X_full_t.std(0, keepdim=True).clamp(min=1e-8)
    X_full_t = (X_full_t - mean_f) / std_f

    Y_full_t  = torch.from_numpy(Y_train.astype(np.float32)).to(DEVICE)
    y_mean_f  = Y_full_t.mean(0, keepdim=True)        # (1, C) — full-train Y mean
    Y_full_c  = Y_full_t - y_mean_f

    N_full = X_full_t.shape[0]
    gamma  = torch.linalg.solve(
        X_full_t @ X_full_t.T + float(best_alpha) * torch.eye(N_full, device=DEVICE),
        Y_full_c,                                     # centered
    )
    beta = X_full_t.T @ gamma

    del X_full_t, Y_full_t, Y_full_c, gamma
    torch.cuda.empty_cache()

    return beta, mean_f, std_f, y_mean_f, best_alpha   # ← NOW RETURNS y_mean_f TOO


def eval_encoding(beta, mean, std, y_mean, X_test, Y_test, nc_flat):
    X_test_t = torch.from_numpy(X_test.astype(np.float32)).to(DEVICE)
    X_test_t = (X_test_t - mean) / std
    with torch.no_grad():
        Y_pred = (X_test_t @ beta + y_mean).cpu().numpy()

    nc_01 = nc_flat / 100.0
    r     = pearson_r_per_unit(Y_test, Y_pred)
    ev    = explained_variance_per_unit(Y_test, Y_pred)
    r_nc  = nc_correct_r(r, nc_01)
    ev_nc = nc_correct_ev(ev, nc_01)

    summary = summarise_metrics(r, r_nc, ev, ev_nc, nc_flat)
    per_channel = {"r": r, "ev": ev, "r_nc": r_nc, "ev_nc": ev_nc}
    return summary, Y_pred, per_channel

# ─────────────────────────────────────────────────────────────
# CELL 2.4.5a  ·  INTERMEDIATE MODE (~10 min run)
# ─────────────────────────────────────────────────────────────
# CELL 2.4.5a — fixed

TEST_MODE = False

if TEST_MODE:
    print("⏳ INTERMEDIATE MODE — N=7500, EEG only")
    N_TRAIN        = 7500
    N_TEST         = 200
    N_LAYERS       = None
    ALPHA_GRID_RUN = np.logspace(0, 7, 15)

    eeg_neural_run = {"occipital_parietal": {
                          "train": eeg_neural["train"][:N_TRAIN],
                          "test":  eeg_neural["test"][:N_TEST]}}
    eeg_nc_run     = {"occipital_parietal": eeg_nc_2d}        # ← 2D (C, T)

    idx_eeg_train_run = feat_idx_eeg_train[:N_TRAIN]
    idx_eeg_test_run  = feat_idx_eeg_test[:N_TEST]

else:
    print("🚀 FULL RUN")
    N_LAYERS       = None
    ALPHA_GRID_RUN = np.logspace(0, 7, 15)

    eeg_neural_run = {"occipital_parietal": eeg_neural}
    eeg_nc_run     = {"occipital_parietal": eeg_nc_2d}        # ← 2D (C, T)

    idx_eeg_train_run = feat_idx_eeg_train
    idx_eeg_test_run  = feat_idx_eeg_test
# ─────────────────────────────────────────────────────────────
# CELL 2.4.5  ·  Main evaluation loop (EEG only)
# ─────────────────────────────────────────────────────────────

import time

def _fmt_time(seconds: float) -> str:
    m, s = divmod(int(seconds), 60)
    return f"{m}m {s:02d}s"

results = []

datasets_cfg = [
    ("EEG2",
     path_a_things, path_b_things,
     idx_eeg_train_run, idx_eeg_test_run,
     eeg_neural_run, eeg_nc_run),
]

rsa_fn = RepresentationalSimilarityAnalysis(dissimilarity="correlation",
                                             similarity_metric="spearman")
cka_fn = CenteredKernelAlignment()

total_jobs = len(list_h5_layers(path_a_things)) * len(eeg_neural_run) * 2
print(f"Total jobs to run: {total_jobs}  (EEG2 × models × layers × targets)\n")

job_done   = 0
wall_start = time.time()

print(f"NC shape: {eeg_nc_run['occipital_parietal'].shape}  (should be (17, 80))")
print(f"Y train shape: {eeg_neural_run['occipital_parietal']['train'].shape}  (should be (16540, 17, 80))")


for (ds_name,
     feat_path_A, feat_path_B,
     idx_train, idx_test,
     neural_dict, nc_dict) in datasets_cfg:

    print(f"\n{'='*60}")
    print(f"  Dataset : {ds_name}")
    print(f"  Targets : {list(neural_dict.keys())}")
    ds_start = time.time()
    print(f"{'='*60}")

    for model_name, feat_path_src in [
        ("Model A (ResNet)", feat_path_A),
        ("Model B (Qwen3)",  feat_path_B),
    ]:
        layers = list_h5_layers(feat_path_src)
        if N_LAYERS:
            layers = layers[:N_LAYERS]
        n_layers = len(layers)
        print(f"\n  ▶ {model_name}  ({n_layers} layers)")
        model_start = time.time()

        for layer_i, layer in enumerate(layers):
            layer_start = time.time()

            X_train = h5_indexed_read(feat_path_src, layer, idx_train)
            X_test  = h5_indexed_read(feat_path_src, layer, idx_test)

            target_scores = []
            for target_key, neural_splits in neural_dict.items():
                # ─── Reshape Y from (N, C, T) → (N, C*T) ───
                Y_train_3d = neural_splits["train"].astype(np.float32)   # (N, C, T)
                Y_test_3d  = neural_splits["test"].astype(np.float32)    # (n_test, C, T)
                N_C, N_T   = Y_train_3d.shape[1], Y_train_3d.shape[2]
            
                Y_train = Y_train_3d.reshape(Y_train_3d.shape[0], N_C * N_T)  # (N, C*T)
                Y_test  = Y_test_3d.reshape(Y_test_3d.shape[0],  N_C * N_T)   # (n_test, C*T)
            
                # ─── Flatten NC from (C, T) → (C*T,) to match ───
                nc_flat = nc_dict[target_key]
                if nc_flat.ndim > 1:
                    nc_flat = nc_flat.flatten()                               # (C*T,)
            
                try:
                    t0 = time.time()
                    beta, mean, std, y_mean, best_alpha = fit_ridge_encoding(
                        X_train, Y_train, alpha_grid=ALPHA_GRID_RUN
                    )
                    fit_t = time.time() - t0
                except Exception as e:
                    print(f"    ⚠ Fit failed [{ds_name}/{target_key}/{layer}]: {e}")
                    job_done += 1
                    continue
            
                metrics, Y_pred, per_channel = eval_encoding(
                    beta, mean, std, y_mean, X_test, Y_test, nc_flat
                )
            
                # ─── Reshape per-output metrics back to (C, T) ───
                r_2d     = per_channel["r"].reshape(N_C, N_T)
                ev_2d    = per_channel["ev"].reshape(N_C, N_T)
                r_nc_2d  = per_channel["r_nc"].reshape(N_C, N_T)
                ev_nc_2d = per_channel["ev_nc"].reshape(N_C, N_T)
            
                # ─── DIAGNOSTIC ───
                print(f"        Y_test  mean={Y_test.mean():.3f}  std={Y_test.std():.3f}")
                print(f"        Y_pred  mean={Y_pred.mean():.3f}  std={Y_pred.std():.3f}")
                print(f"        Y_train mean={Y_train.mean():.3f}  std={Y_train.std():.3f}")
                print(f"        Pearson r (mean over C×T):       {r_2d.mean():.3f}")
                print(f"        Pearson r (mean over channels at peak ~150ms): {r_2d[:, min(15, N_T-1)].mean():.3f}")
                print(f"        EV       (mean over C×T):       {ev_2d.mean():.3f}")
                # ─────────────────
            
                # RSA / CKA on the flattened (C*T) representation — comparable across layers
                try:
                    enc_rsa = rsa_fn(Y_pred, Y_test)
                except Exception:
                    enc_rsa = float("nan")
                try:
                    enc_cka = cka_fn(Y_pred, Y_test)
                except Exception:
                    enc_cka = float("nan")
            
                results.append({
                    "dataset":           ds_name,
                    "model":             model_name,
                    "layer":             layer,
                    "target":            target_key,
                    "alpha":             best_alpha,
                    **metrics,                          # scalar means over C*T
                    "r_per_ch_t":        r_2d,          # (C, T)
                    "ev_per_ch_t":       ev_2d,         # (C, T)
                    "r_nc_per_ch_t":     r_nc_2d,       # (C, T)
                    "ev_nc_per_ch_t":    ev_nc_2d,      # (C, T)
                    "encoding_rsa":      enc_rsa,
                    "encoding_cka":      enc_cka,
                })
            
                target_scores.append(
                    f"{target_key.split('/')[-1]}  "
                    f"r={metrics['pearsonr']:.3f}  "
                    f"r_nc={metrics['pearsonr_nc']:.3f}  "
                    f"α={best_alpha:.0e}  "
                    f"fit={fit_t:.1f}s"
                )
                job_done += 1

            layer_elapsed = time.time() - layer_start
            elapsed_total = time.time() - wall_start
            eta = (elapsed_total / max(job_done, 1)) * (total_jobs - job_done)

            layer_short = layer.split("/")[-1]
            scores_str  = " | ".join(target_scores) if target_scores else "no targets"
            print(
                f"    [{layer_i+1:2d}/{n_layers}] {layer_short:<28} "
                f"{scores_str}   "
                f"(layer {layer_elapsed:.1f}s | "
                f"elapsed {_fmt_time(elapsed_total)} | "
                f"ETA {_fmt_time(eta)})"
            )
            print(f"        Pearson r (mean over C*T): {r_2d.mean():.3f}")
            print(f"        Pearson r at peak ~100ms : {r_2d[:, 10].mean():.3f}")

        model_elapsed = time.time() - model_start
        print(f"\n  ✓ {model_name} done in {_fmt_time(model_elapsed)}")

    ds_elapsed = time.time() - ds_start
    print(f"\n  ✓✓ {ds_name} complete in {_fmt_time(ds_elapsed)}")

total_elapsed = time.time() - wall_start
df_results = pd.DataFrame(results)
print(f"\n{'='*60}")
print(f"✅  All done in {_fmt_time(total_elapsed)} — "
      f"{len(df_results)} rows in results table.")
print(f"{'='*60}")
df_results.head()


In [ ]:
print(df_results.shape)
print(df_results.columns.tolist())
print(df_results.head())

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 2.4.X · Time-resolved encoding plots (Part 3 figures)
# ─────────────────────────────────────────────────────────────
import re
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# ── Setup ──────────────────────────────────────────────────
NC_THRESHOLD = 0.10
nc_ct = eeg_nc_2d / 100.0           # (C, T) in (0, 1)
valid = nc_ct >= NC_THRESHOLD       # (C, T) bool — prof's filter
N_T = nc_ct.shape[1]
EEG_FS = 100  # Hz
time_ms = np.arange(N_T) * (1000 / EEG_FS)   # 0, 10, …, 790 ms

def natkey(s):
    return [int(t) if t.isdigit() else t.lower()
            for t in re.split(r"(\d+)", s)]

def time_curve(r_ct, valid_mask):
    """Mean r across valid channels at each timepoint → (T,)."""
    masked = np.where(valid_mask, r_ct, np.nan)
    return np.nanmean(masked, axis=0)

# ── Per model: build (n_layers × T) matrix of mean-r-over-channels ──
model_curves = {}
for model_name in df_results["model"].unique():
    sub = df_results[df_results["model"] == model_name].copy()
    sub = sub.iloc[sub["layer"].map(natkey).argsort()].reset_index(drop=True)
    R_lt = np.stack([time_curve(row["r_per_ch_t"], valid)
                     for _, row in sub.iterrows()])    # (L, T)
    model_curves[model_name] = {
        "R_lt":   R_lt,
        "layers": sub["layer"].values,
    }

# Also: NC curve across channels (mean over valid channels per timepoint)
nc_curve = np.where(valid, nc_ct, np.nan)
nc_curve = np.nanmean(nc_curve, axis=0)             # (T,)
nc_ceiling_r = np.sqrt(np.clip(nc_curve, 0, None))  # max achievable r per t


# ============================================================
# FIGURE 1 — Time-resolved r curves, one line per layer
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, (model_name, data) in zip(axes, model_curves.items()):
    R_lt   = data["R_lt"]
    layers = data["layers"]
    cmap   = plt.cm.viridis(np.linspace(0.05, 0.95, len(layers)))

    for i, (layer, r_t) in enumerate(zip(layers, R_lt)):
        ax.plot(time_ms, r_t, color=cmap[i], linewidth=1.6,
                label=layer.split("/")[-1], alpha=0.85)

    # Noise ceiling envelope
    ax.plot(time_ms, nc_ceiling_r, color="black", linewidth=1.5,
            linestyle="--", label="Noise ceiling (√NC)", alpha=0.7)

    ax.axhline(0, color="gray", linewidth=0.5)
    ax.set_xlabel("Time post-stimulus (ms)")
    ax.set_title(model_name)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=6.5, loc="upper right", ncol=2)

axes[0].set_ylabel("Pearson r (mean across valid channels)")
fig.suptitle("Time-resolved encoding performance per layer", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


# ============================================================
# FIGURE 2 — (Layer × Time) heatmaps  — FIXED
# ============================================================
import warnings

# Robust shared scale: 1st–99th percentile across both models, ignoring NaNs
all_vals = np.concatenate([d["R_lt"].ravel() for d in model_curves.values()])
all_vals = all_vals[np.isfinite(all_vals)]   # drop NaN and inf
vmin, vmax = np.percentile(all_vals, [1, 99])
vmax = max(vmax, 0.1)                         # don't let it collapse if data is small
vmin = min(vmin, 0.0)

print(f"Heatmap colour range: vmin={vmin:.3f}, vmax={vmax:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (model_name, data) in zip(axes, model_curves.items()):
    R_lt   = data["R_lt"]
    layers = data["layers"]

    print(f"{model_name}: R_lt min={np.nanmin(R_lt):.3f}, "
          f"max={np.nanmax(R_lt):.3f}, "
          f"any NaN={np.any(np.isnan(R_lt))}, "
          f"any inf={np.any(np.isinf(R_lt))}")

    im = ax.imshow(R_lt, aspect="auto", cmap="viridis",
                   vmin=vmin, vmax=vmax,
                   extent=[time_ms[0], time_ms[-1],
                           len(layers) - 0.5, -0.5],
                   interpolation="nearest")
    ax.set_yticks(range(len(layers)))
    ax.set_yticklabels([l.split("/")[-1] for l in layers], fontsize=7)
    ax.set_xlabel("Time post-stimulus (ms)")
    ax.set_title(model_name)

axes[0].set_ylabel("Layer (early → late)")
fig.colorbar(im, ax=axes, label="Pearson r", fraction=0.025, pad=0.02)
fig.suptitle("Encoding r — layer × time", fontsize=12, y=1.02)
plt.show()


# ============================================================
# FIGURE 3 — Best-layer-per-timepoint (the H1 test)  — FIXED
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (model_name, data) in zip(axes, model_curves.items()):
    R_lt   = data["R_lt"]                              # (L, T)
    layers = data["layers"]
    n_L    = len(layers)

    # Skip timepoints where every layer is NaN (no valid channels at that t)
    has_signal = ~np.all(np.isnan(R_lt), axis=0)       # (T,) bool

    best_layer_idx = np.full(R_lt.shape[1], -1)
    best_r         = np.full(R_lt.shape[1], np.nan)
    best_layer_idx[has_signal] = np.nanargmax(R_lt[:, has_signal], axis=0)
    best_r[has_signal]         = np.nanmax(R_lt[:, has_signal], axis=0)

    # Only show timepoints where there's meaningful signal
    sig_mask = has_signal & (best_r > 0.05)
    t_sig     = time_ms[sig_mask]
    layer_sig = best_layer_idx[sig_mask]

    # Spearman correlation between time and best-layer-index
    if len(t_sig) > 5:
        rho, p = spearmanr(t_sig, layer_sig)
    else:
        rho, p = float("nan"), float("nan")

    sc = ax.scatter(t_sig, layer_sig, c=best_r[sig_mask],
                    cmap="viridis", s=40, edgecolor="white", linewidth=0.5)
    ax.set_xlabel("Time post-stimulus (ms)")
    ax.set_yticks(range(n_L))
    ax.set_yticklabels([l.split("/")[-1] for l in layers], fontsize=7)
    ax.set_title(f"{model_name}\nSpearman ρ(time, best layer) = {rho:.2f}  (p = {p:.3g})")
    ax.grid(alpha=0.3)
    ax.set_xlim(time_ms[0], time_ms[-1])
    ax.set_ylim(-0.5, n_L - 0.5)

axes[0].set_ylabel("Best-predicting layer (early → late)")
fig.suptitle("H1 test: does the best-predicting layer shift later in time?",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

# ============================================================
# FIGURE 4 — Baseline comparison: time-resolved vs scalar
# ============================================================
# This addresses the TA's request: "one direct comparison against the linear baseline"
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, (model_name, data) in zip(axes, model_curves.items()):
    R_lt   = data["R_lt"]
    layers = data["layers"]

    # Two summaries per layer:
    #   (a) Mean r over all valid timepoints (= the "baseline" scalar score)
    #   (b) Peak r across timepoints (= the "best moment" the time-resolved
    #       readout uncovers)
    mean_r = np.nanmean(R_lt, axis=1)
    peak_r = np.nanmax(R_lt,  axis=1)

    x = np.arange(len(layers))
    ax.bar(x - 0.2, mean_r, width=0.4, label="Time-averaged (baseline-equivalent)",
           color="#4c78a8", edgecolor="white")
    ax.bar(x + 0.2, peak_r, width=0.4, label="Peak across time (time-resolved)",
           color="#f58518", edgecolor="white")
    ax.set_xticks(x)
    ax.set_xticklabels([l.split("/")[-1] for l in layers],
                       rotation=45, ha="right", fontsize=7)
    ax.set_title(model_name)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(fontsize=8)

axes[0].set_ylabel("Pearson r")
fig.suptitle("Time-resolved readout uncovers higher per-layer performance than scalar averaging",
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL · Time-resolved explained variance plots
# ─────────────────────────────────────────────────────────────

# Build (n_layers × T) matrices for EV and EV_nc, mean over valid channels
def build_ev_curves(metric_key):
    out = {}
    for model_name in df_results["model"].unique():
        sub = df_results[df_results["model"] == model_name].copy()
        sub = sub.iloc[sub["layer"].map(natkey).argsort()].reset_index(drop=True)
        M_lt = np.stack([time_curve(row[metric_key], valid)
                         for _, row in sub.iterrows()])    # (L, T)
        out[model_name] = {
            "M_lt":   M_lt,
            "layers": sub["layer"].values,
        }
    return out

ev_curves    = build_ev_curves("ev_per_ch_t")
ev_nc_curves = build_ev_curves("ev_nc_per_ch_t")

# ============================================================
# FIGURE 5 — Time-resolved EV curves, one line per layer
# ============================================================
import warnings

for curves_dict, label, suptitle in [
    (ev_curves,    "Explained variance",                "Time-resolved EV per layer"),
    (ev_nc_curves, "Explained variance (NC-corrected)", "Time-resolved EV (NC-corrected) per layer"),
]:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    for ax, (model_name, data) in zip(axes, curves_dict.items()):
        M_lt   = data["M_lt"]
        layers = data["layers"]
        cmap   = plt.cm.viridis(np.linspace(0.05, 0.95, len(layers)))

        for i, (layer, m_t) in enumerate(zip(layers, M_lt)):
            ax.plot(time_ms, m_t, color=cmap[i], linewidth=1.6,
                    label=layer.split("/")[-1], alpha=0.85)

        ax.axhline(0, color="gray", linewidth=0.5)
        ax.set_xlabel("Time post-stimulus (ms)")
        ax.set_title(model_name)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=6.5, loc="upper right", ncol=2)

    axes[0].set_ylabel(f"{label} (mean across valid channels)")
    
    # --- ADD THIS TO FIX THE SCALE ---
    if "NC-corrected" in label:
        # Prevent the massive negative spikes from ruining the scale
        # Adjust the top limit (e.g., 0.8) based on your highest positive EV_nc
        axes[0].set_ylim(-0.5, 0.8) 
    else:
        # Keep the raw EV scale tight too, if desired
        axes[0].set_ylim(-0.5, 0.4) 
    # ---------------------------------

    fig.suptitle(suptitle, fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()

# ============================================================
# FIGURE 6 — (Layer × Time) heatmaps for EV and EV_nc
# ============================================================
for curves_dict, label, suptitle in [
    (ev_curves,    "EV",                "Encoding EV — layer × time"),
    (ev_nc_curves, "EV (NC-corrected)", "Encoding EV (NC-corrected) — layer × time"),
]:
    # Robust shared scale across both models
    all_vals = np.concatenate([d["M_lt"].ravel() for d in curves_dict.values()])
    all_vals = all_vals[np.isfinite(all_vals)]
    vmin, vmax = np.percentile(all_vals, [1, 99])
    print(f"{label}: vmin={vmin:.3f}, vmax={vmax:.3f}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, (model_name, data) in zip(axes, curves_dict.items()):
        M_lt   = data["M_lt"]
        layers = data["layers"]

        im = ax.imshow(M_lt, aspect="auto", cmap="viridis",
                       vmin=vmin, vmax=vmax,
                       extent=[time_ms[0], time_ms[-1],
                               len(layers) - 0.5, -0.5],
                       interpolation="nearest")
        ax.set_yticks(range(len(layers)))
        ax.set_yticklabels([l.split("/")[-1] for l in layers], fontsize=7)
        ax.set_xlabel("Time post-stimulus (ms)")
        ax.set_title(model_name)

    axes[0].set_ylabel("Layer (early → late)")
    fig.colorbar(im, ax=axes, label=label, fraction=0.025, pad=0.02)
    fig.suptitle(suptitle, fontsize=12, y=1.02)
    plt.show()


# ============================================================
# FIGURE 7 — Baseline comparison for EV
# ============================================================
for curves_dict, label, suptitle in [
    (ev_curves,    "EV",                "Time-resolved EV: averaged vs peak"),
    (ev_nc_curves, "EV (NC-corrected)", "Time-resolved EV (NC-corrected): averaged vs peak"),
]:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
    for ax, (model_name, data) in zip(axes, curves_dict.items()):
        M_lt   = data["M_lt"]
        layers = data["layers"]

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            mean_m = np.nanmean(M_lt, axis=1)
            peak_m = np.nanmax(M_lt,  axis=1)

        x = np.arange(len(layers))
        ax.bar(x - 0.2, mean_m, width=0.4, label="Time-averaged",
               color="#4c78a8", edgecolor="white")
        ax.bar(x + 0.2, peak_m, width=0.4, label="Peak across time",
               color="#f58518", edgecolor="white")
        ax.set_xticks(x)
        ax.set_xticklabels([l.split("/")[-1] for l in layers],
                           rotation=45, ha="right", fontsize=7)
        ax.set_title(model_name)
        ax.grid(axis="y", alpha=0.3)
        ax.axhline(0, color="black", linewidth=0.5)
        ax.legend(fontsize=8)

    axes[0].set_ylabel(label)
    fig.suptitle(suptitle, fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()